# Delivery Time Prediction - Phase 4: Data Preprocessing

## Executive Overview
Welcome to **Phase 4: Data Preprocessing** of the Delivery Time Prediction machine learning project.

In **Phase 2 (Dataset Understanding)** and **Phase 3 (Exploratory Data Analysis)**, we profiled the dataset schema, assessed distributions, and identified critical drivers of delivery latency. 

The goal of this phase is to construct a **robust, leak-free, modular Scikit-Learn preprocessing pipeline** that handles missing values, encodes categorical variables, and scales numerical features.

### Objectives of this Phase
1. Load the original dataset while ensuring zero modification to the raw CSV.
2. Audit duplicates and verify data consistency.
3. Formulate and implement principled missing value imputation.
4. Formalize outlier decisions based on Phase 3 findings.
5. Separate predictors ($X$) from the target variable ($Y = \text{Delivery\_Time\_min}$) and remove arbitrary identifiers (`Order_ID`).
6. Construct specialized sub-pipelines for numerical, ordinal, and nominal features using `ColumnTransformer`.
7. Demonstrate pipeline fitting, inspect transformed feature names, and verify absence of missing values.
8. Serialize the preprocessing pipeline for production serving and reproducible training.
9. Enforce data leakage safeguards for downstream train/test splitting.

> **Project Rules**:
> * No feature engineering (e.g. creating polynomial or interaction features).
> * No model training or evaluation.
> * The raw dataset remains strictly immutable.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

# Display configuration
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

# Load dataset
DATA_PATH = os.path.join('..', 'data', 'delivery_data.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join('..', 'data', 'Food_Delivery_Times.csv')

df = pd.read_csv(DATA_PATH)
print(f"Dataset successfully loaded from: '{DATA_PATH}'")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

Dataset successfully loaded from: '..\data\delivery_data.csv'
Shape: 1000 rows x 9 columns


### Analysis & Observations
* The dataset was loaded directly into a Pandas DataFrame.
* The original CSV file on disk is read in read-only mode and remains strictly untouched.

## 1. Data Integrity Audit: Duplicates & Inconsistencies
We verify that no duplicate records or corrupted values exist before assembling transformations.

In [2]:
# Check duplicate rows
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows detected: {n_duplicates}")

# Check unique Order_ID integrity
n_duplicate_ids = df['Order_ID'].duplicated().sum()
print(f"Duplicate Order_IDs detected: {n_duplicate_ids}")

# Verify non-negativity across numeric fields
negative_violations = {col: (df[col] < 0).sum() for col in ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time_min']}
print("Negative value counts:", negative_violations)

Duplicate rows detected: 0
Duplicate Order_IDs detected: 0
Negative value counts: {'Distance_km': np.int64(0), 'Preparation_Time_min': np.int64(0), 'Courier_Experience_yrs': np.int64(0), 'Delivery_Time_min': np.int64(0)}


### Preprocessing Decision: Duplicates & Integrity
* **Finding**: 0 duplicate rows, 0 duplicate `Order_ID`s, and 0 negative values.
* **Action**: No row deduplication or value clipping is required. All records are valid delivery transactions.

## 2. Missing Value Imputation Strategy
From Phase 2 and 3, exactly 4 columns contain missing values (30 nulls each, 3.0% per column):
* `Courier_Experience_yrs` (Numerical)
* `Weather` (Nominal Categorical)
* `Traffic_Level` (Ordinal Categorical)
* `Time_of_Day` (Nominal Categorical)

### Principles for Imputation:
1. **`Courier_Experience_yrs`**:
   * **Method**: `SimpleImputer(strategy='median')`
   * **Rationale**: The median experience is 5.0 years. Median is resilient to slight skewness and preserves integer year boundaries without introducing floating-point estimation bias.
2. **Categorical Features (`Weather`, `Traffic_Level`, `Time_of_Day`)**:
   * **Method**: `SimpleImputer(strategy='most_frequent')`
   * **Rationale**: Missingness is low (3%). Mode imputation (`Clear` for weather, `Medium` for traffic, `Morning` for time of day) assigns the most probable historical category without introducing synthetic labels or increasing feature dimensionality unnecessarily.

In [3]:
# Display current missing values summary
null_summary = df.isnull().sum()
print("Missing values per column before preprocessing:")
print(null_summary[null_summary > 0])

Missing values per column before preprocessing:
Weather                   30
Traffic_Level             30
Time_of_Day               30
Courier_Experience_yrs    30
dtype: int64


## 3. Outlier Handling Decision
In Phase 3, 6 records were flagged as having `Delivery_Time_min > 116.0 min` using the $1.5 \times \text{IQR}$ rule: `[122, 123, 126, 141, 141, 153]`.

### Preprocessing Decision:
* **Inspection**: All 6 orders have long distances (14.8 km to 19.0 km) and high preparation times (19 to 29 min).
* **Verdict**: These are **legitimate long-distance delivery orders**, not recording errors or corrupted sensor data.
* **Action**: **Retain all 6 records**. Dropping or clamping these valid orders would artificially truncate the model's capacity to predict realistic delivery times for outer-suburb or cross-town deliveries.

## 4. Separating Features ($X$) and Target ($Y$)
We isolate the target variable `Delivery_Time_min` and drop the arbitrary transactional identifier `Order_ID`.

In [4]:
# Define target and identifier columns
TARGET_COL = 'Delivery_Time_min'
ID_COL = 'Order_ID'

# Separate features (X) and target (y)
X = df.drop(columns=[ID_COL, TARGET_COL])
y = df[TARGET_COL]

print(f"Features (X) shape: {X.shape[0]} rows x {X.shape[1]} columns")
print(f"Target (y) shape:   {y.shape[0]} elements")
print(f"\nInput Features: {list(X.columns)}")

Features (X) shape: 1000 rows x 7 columns
Target (y) shape:   1000 elements

Input Features: ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']


### Analysis & Observations
* **Exclusion of `Order_ID`**: `Order_ID` is a monotonically assigned transaction number with zero predictive correlation. Removing it prevents spurious memorization and target leakage.
* **Feature Count**: $X$ contains 7 raw candidate features (3 numerical, 1 ordinal, 3 nominal).
* **Target ($Y$)**: Continuous delivery time in minutes.

## 5. Designing the Preprocessing Pipelines
We construct modular, isolated Scikit-Learn pipelines for each feature modality:

### 1. Numerical Pipeline
* **Features**: `Distance_km`, `Preparation_Time_min`, `Courier_Experience_yrs`
* **Steps**:
  1. `SimpleImputer(strategy='median')`: Fills missing courier experience with median.
  2. `StandardScaler()`: Standardizes features to zero mean and unit variance ($z = \frac{x - \mu}{\sigma}$).
* **Why Scaling?**: Distance (0.5–20 km), prep time (5–29 min), and experience (0–9 yrs) have different measurement units. Scaling is required for distance-based and regularized linear models (Ridge, Lasso, ElasticNet, KNN, Neural Nets).

### 2. Ordinal Pipeline
* **Features**: `Traffic_Level`
* **Steps**:
  1. `SimpleImputer(strategy='most_frequent')`: Fills missing traffic with the mode (`Medium`).
  2. `OrdinalEncoder(categories=[['Low', 'Medium', 'High']])`: Encodes explicit natural ordering ($Low = 0, Medium = 1, High = 2$).
* **Why Ordinal?**: Phase 3 demonstrated a strictly monotonic delay penalty as traffic worsens (+12 min in high traffic). Ordinal encoding captures this monotonic rank without expanding columns.

### 3. Nominal Pipeline
* **Features**: `Weather`, `Time_of_Day`, `Vehicle_Type`
* **Steps**:
  1. `SimpleImputer(strategy='most_frequent')`: Fills missing weather/time with the mode.
  2. `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`: Creates binary dummy indicators for unordered categories.
* **Why One-Hot?**: None of these features have a natural mathematical ordering. `handle_unknown='ignore'` guarantees resilience against unseen categories during inference.

In [5]:
# Define column groupings
numerical_features = ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs']
ordinal_features = ['Traffic_Level']
nominal_features = ['Weather', 'Time_of_Day', 'Vehicle_Type']

# 1. Numerical Pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Ordinal Pipeline
ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['Low', 'Medium', 'High']]))
])

# 3. Nominal Pipeline
nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine all branches into a unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_features),
        ('ord', ord_pipeline, ordinal_features),
        ('nom', nom_pipeline, nominal_features)
    ],
    remainder='drop'
)

print("ColumnTransformer architecture successfully defined:")
print(preprocessor)

ColumnTransformer architecture successfully defined:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Distance_km', 'Preparation_Time_min',
                                  'Courier_Experience_yrs']),
                                ('ord',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['Low',
                                                                              'Medium',
                                                                              'High']]))]),
                          

## 6. Fitting and Transforming Features
We fit the preprocessor on $X$ and transform the data into a dense, clean numerical matrix.

In [6]:
# Fit and transform features
X_transformed = preprocessor.fit_transform(X)

# Retrieve transformed column names
feature_names = preprocessor.get_feature_names_out()

# Create a clean DataFrame of transformed features for inspection
X_processed_df = pd.DataFrame(X_transformed, columns=feature_names)

print(f"Transformed feature matrix shape: {X_processed_df.shape}")
print(f"Total features created: {len(feature_names)}")
print(f"Any missing values remaining?: {X_processed_df.isnull().any().any()}")
print("\n--- First 3 Processed Records ---")
X_processed_df.head(3).T

Transformed feature matrix shape: (1000, 16)
Total features created: 16
Any missing values remaining?: False

--- First 3 Processed Records ---


,0,1,2
num__Distance_km,-0.374085,1.117008,-0.094835
num__Preparation_Time_min,-0.691853,0.419111,1.530076
num__Courier_Experience_yrs,-1.251672,-0.903211,-1.251672
ord__Traffic_Level,0.000000,1.000000,0.000000
nom__Weather_Clear,0.000000,1.000000,0.000000
nom__Weather_Foggy,0.000000,0.000000,1.000000
nom__Weather_Rainy,0.000000,0.000000,0.000000
nom__Weather_Snowy,0.000000,0.000000,0.000000
nom__Weather_Windy,1.000000,0.000000,0.000000
nom__Time_of_Day_Afternoon,1.000000,0.000000,0.000000


### Analysis & Observations
* **Dimensionality**: The 7 raw features are transformed into **16 clean, numerical features**:
  * 3 standardized numerical features (`num__Distance_km`, `num__Preparation_Time_min`, `num__Courier_Experience_yrs`)
  * 1 rank-encoded ordinal feature (`ord__Traffic_Level` $\in \{0, 1, 2\}$)
  * 5 one-hot weather features (`nom__Weather_Clear`, `Foggy`, `Rainy`, `Snowy`, `Windy`)
  * 4 one-hot time of day features (`nom__Time_of_Day_Afternoon`, `Evening`, `Morning`, `Night`)
  * 3 one-hot vehicle features (`nom__Vehicle_Type_Bike`, `Car`, `Scooter`)
* **Zero Missing Values**: Missing value audit confirms **0 nulls remaining** across the entire matrix.
* **Standardization Check**: Continuous features have Mean $\approx 0.0$ and Standard Deviation $= 1.0$.

## 7. Saving the Preprocessing Pipeline
We serialize the preprocessor object to disk using `joblib` in the `models/` directory for seamless downstream training and API inference.

In [7]:
# Ensure models directory exists
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

pipeline_path = os.path.join(models_dir, 'preprocessor.joblib')
joblib.dump(preprocessor, pipeline_path)
print(f"Preprocessing pipeline successfully saved to: '{pipeline_path}'")

# Test reloading the pipeline
loaded_preprocessor = joblib.load(pipeline_path)
test_transform = loaded_preprocessor.transform(X.head(5))
print(f"Reload verification successful! Test transform shape: {test_transform.shape}")

Preprocessing pipeline successfully saved to: '..\models\preprocessor.joblib'
Reload verification successful! Test transform shape: (5, 16)


## 8. Data Leakage Protocol & Phase 5 Readiness

### Critical Machine Learning Safeguard:
> **Important Rule for Phase 5 (Train/Test Split & Modeling)**:
> In this Phase 4 demonstration, we fitted the pipeline on the full dataset to verify transformed shapes, feature names, and pipeline serialization.
> 
> When the official train/test split is introduced in **Phase 5**:
> 1. `X_train` and `X_test` will be split **FIRST**.
> 2. `preprocessor.fit_transform(X_train)` will learn imputation medians, scaler means/variances, and category encoders **ONLY from `X_train`**.
> 3. `preprocessor.transform(X_test)` will strictly apply the frozen training parameters onto `X_test`.
> 4. This guarantees zero data leakage from the test distribution into the training pipeline.

### Summary of Preprocessing Decisions:
| Component | Decision | Rationale |
| :--- | :--- | :--- |
| **`Order_ID`** | Dropped | Non-informative arbitrary identifier. |
| **Duplicates** | None removed | 0 duplicates found in data. |
| **Outliers** | Retained | 6 orders > 116 min are valid long-distance deliveries. |
| **`Courier_Experience_yrs`** | Median Imputation | Resilient to skewness; maintains realistic year values. |
| **`Weather`, `Traffic_Level`, `Time_of_Day`** | Mode Imputation | Low missingness (3%); avoids unnecessary synthetic categories. |
| **`Traffic_Level`** | Ordinal Encoding | Exploits natural monotonic delay progression ($Low=0, Med=1, High=2$). |
| **`Weather`, `Time_of_Day`, `Vehicle_Type`** | One-Hot Encoding | Nominal unordered categories; handles unknown inputs safely. |
| **Numerical Features** | StandardScaler | Centers and standardizes diverse scales (km, min, yrs). |

*Phase 4: Data Preprocessing is concluded. Ready for Phase 5: Train/Test Split.*